# 02 Major Index Econometric Benchmarks

This notebook runs the econometric benchmark suite on the shared dataset using a single-fit chronological design.

## Forecast Target
- One-step-ahead daily return
- Training sample: target dates in `2022`
- Evaluation sample: target dates in `2023-01-01` to `2026-03-31`

## Benchmark Design
- `LinearRegression`
- `ARMA(0,1)`, `ARMA(1,0)`, `ARMA(1,1)`, `ARMA(1,2)`, `ARMA(2,1)`, `ARMA(2,2)`
- `ARIMA(0,1,0)`, `ARIMA(0,1,1)`, `ARIMA(1,1,0)`, `ARIMA(1,1,1)`, `ARIMA(1,1,2)`, `ARIMA(2,1,1)`, `ARIMA(2,1,2)`
- `ARCH(1)`, `ARCH(2)`, `ARCH(5)`, `ARCH(10)`
- `GARCH(1,1)`, `GARCH(1,2)`, `GARCH(2,1)`, `GARCH(2,2)`
- `GJR-GARCH(1,1)` and `EGARCH(1,1)`

## Rationale
- This notebook fits each benchmark once on the `2022` training period and evaluates on `2023-2026`
- The benchmark notebook therefore avoids the expensive rolling refit loop used by the ML, deep-learning, and quantum notebooks


In [ ]:
from __future__ import annotations

import hashlib
import json
import time
import warnings
from contextlib import contextmanager
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA
from tqdm.auto import tqdm

try:
    from arch import arch_model
    ARCH_AVAILABLE = True
except Exception:
    arch_model = None
    ARCH_AVAILABLE = False


def resolve_project_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Run this notebook from the project root or its notebooks directory.")


PROJECT_ROOT = resolve_project_root()
PIPELINE_BUNDLE_PATH = PROJECT_ROOT / "outputs" / "01_major_index_sliding_window_pipeline" / "major_index_sliding_window_bundle.joblib"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "02_major_index_econometric_benchmarks"
LOG_DIR = OUTPUT_ROOT / "rolling_logs"
PREDICTIONS_DIR = OUTPUT_ROOT / "predictions"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIG_DIR = OUTPUT_ROOT / "figures"
for directory in [OUTPUT_ROOT, LOG_DIR, PREDICTIONS_DIR, TABLE_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

TRAIN_END_DATE = "2022-12-31"
EVAL_START_DATE = "2023-01-01"
TARGET_COL = "y"
FEATURE_COLS = ["close", "rsi14", "macd_hist", "bb_z20", "roc10", "vol20", "sma10_gap", "atr14_pct", "williams_r14"]
NOTEBOOK_TAG = "02_major_index_econometric_benchmarks"
ALL_LOGS_PATH = OUTPUT_ROOT / "all_prediction_logs.csv"
SCORES_SUMMARY_PATH = TABLE_DIR / "scores_summary.csv"
BEST_CONFIGS_PATH = TABLE_DIR / "best_configs.csv"
MODEL_SUMMARY_PATH = TABLE_DIR / "model_summary.csv"
TIMING_SUMMARY_PATH = TABLE_DIR / "timing_summary.csv"

TIMING_ROWS: list[dict] = []
FORCE_FRESH_RUN = False


In [ ]:
@contextmanager
def timed_step(step_name: str):
    start = time.perf_counter()
    yield
    elapsed_seconds = time.perf_counter() - start
    TIMING_ROWS.append({"step": step_name, "elapsed_seconds": elapsed_seconds})
    print(f"[TIMER] {step_name}: {elapsed_seconds:.2f}s")


def load_index_frames(path: Path = PIPELINE_BUNDLE_PATH) -> dict[str, pd.DataFrame]:
    bundle = joblib.load(path)["case_bundle"]
    frames: dict[str, pd.DataFrame] = {}
    for case in bundle.values():
        index_key = str(case["index_key"])
        if index_key not in frames:
            frame = case["df"].copy()
            frame["Date"] = pd.to_datetime(frame["Date"], errors="coerce")
            frame["target_date"] = pd.to_datetime(frame["target_date"], errors="coerce")
            frames[index_key] = frame.sort_values("Date").reset_index(drop=True)
    return frames


def split_train_eval(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_df = frame.loc[frame["target_date"] <= pd.Timestamp(TRAIN_END_DATE)].copy().reset_index(drop=True)
    eval_df = frame.loc[frame["target_date"] >= pd.Timestamp(EVAL_START_DATE)].copy().reset_index(drop=True)
    return train_df, eval_df


LOG_DATE_COLS = ["test_reference_date", "test_target_date"]
SCORE_KEY_COLS = ["index_key", "model", "config_index"]
PREDICTION_KEY_COLS = ["index_key", "model_name", "config_index", "test_target_date"]


def reset_output_targets() -> None:
    for path in [ALL_LOGS_PATH, SCORES_SUMMARY_PATH, BEST_CONFIGS_PATH, MODEL_SUMMARY_PATH, TIMING_SUMMARY_PATH]:
        if path.exists():
            path.unlink()
    for directory in [LOG_DIR, PREDICTIONS_DIR]:
        for csv_path in directory.glob("*.csv"):
            csv_path.unlink()
    for png_path in FIG_DIR.glob("*.png"):
        png_path.unlink()


def append_frame(path: Path, frame: pd.DataFrame) -> None:
    if frame is None or frame.empty:
        return
    frame.to_csv(path, mode="a", header=not path.exists(), index=False)


def load_progress_frame(path: Path, key_cols: list[str] | None = None) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path)
    for col in LOG_DATE_COLS:
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col], errors="coerce")
    if key_cols and all(col in frame.columns for col in key_cols):
        frame = frame.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)
    return frame


def completed_config_keys(frame: pd.DataFrame) -> set[tuple[str, str, int]]:
    if frame.empty or "status" not in frame.columns:
        return set()
    completed = frame.loc[frame["status"].eq("completed")].copy()
    if completed.empty:
        return set()
    return {
        (str(row["index_key"]), str(row["model"]), int(row["config_index"]))
        for _, row in completed.iterrows()
    }


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    yt = np.asarray(y_true, dtype=float)[mask]
    yp = np.asarray(y_pred, dtype=float)[mask]
    if len(yt) == 0:
        return {"MAE": np.nan, "RMSE": np.nan, "R2": np.nan, "n_points": 0}
    return {
        "MAE": float(mean_absolute_error(yt, yp)),
        "RMSE": float(np.sqrt(mean_squared_error(yt, yp))),
        "R2": float(r2_score(yt, yp)) if len(yt) > 1 else np.nan,
        "n_points": int(len(yt)),
    }


def directional_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(yt) & np.isfinite(yp)
    if int(mask.sum()) == 0:
        return np.nan
    return float(np.mean(np.sign(yt[mask]) == np.sign(yp[mask])))


def summarize_config(config: dict) -> str:
    if not config:
        return "default"
    return " ".join(f"{key}={value}" for key, value in config.items())


def config_key(model_name: str, config: dict) -> str:
    payload = json.dumps({"model": model_name, **config}, sort_keys=True)
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()[:16]


def fit_linear_model(train_df: pd.DataFrame, eval_df: pd.DataFrame) -> dict:
    X_train = train_df[FEATURE_COLS].to_numpy(dtype=float)
    y_train = train_df[TARGET_COL].to_numpy(dtype=float)
    X_eval = eval_df[FEATURE_COLS].to_numpy(dtype=float)
    x_scaler = StandardScaler().fit(X_train)
    y_scaler = StandardScaler().fit(y_train.reshape(-1, 1))
    X_train_s = x_scaler.transform(X_train)
    X_eval_s = x_scaler.transform(X_eval)
    y_train_s = y_scaler.transform(y_train.reshape(-1, 1)).ravel()
    model = LinearRegression()
    fit_start = time.perf_counter()
    model.fit(X_train_s, y_train_s)
    train_pred = y_scaler.inverse_transform(model.predict(X_train_s).reshape(-1, 1)).ravel()
    eval_pred = y_scaler.inverse_transform(model.predict(X_eval_s).reshape(-1, 1)).ravel()
    fit_seconds = time.perf_counter() - fit_start
    train_metrics = regression_metrics(y_train, train_pred)
    return {
        "y_train_pred": train_pred,
        "y_eval_pred": eval_pred,
        "selection_metric_name": "train_RMSE",
        "selection_metric": train_metrics["RMSE"],
        "fit_seconds": float(fit_seconds),
        "aic": np.nan,
    }


def fit_arima_model(train_df: pd.DataFrame, eval_df: pd.DataFrame, order: tuple[int, int, int]) -> dict:
    series = pd.Series(train_df[TARGET_COL].to_numpy(dtype=float))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fit_start = time.perf_counter()
        fitted = ARIMA(series, order=order, enforce_stationarity=False, enforce_invertibility=False).fit()
        fit_seconds = time.perf_counter() - fit_start
        train_pred = np.asarray(fitted.fittedvalues, dtype=float).reshape(-1)
        eval_pred = np.asarray(fitted.forecast(steps=len(eval_df)), dtype=float).reshape(-1)
    return {
        "y_train_pred": train_pred,
        "y_eval_pred": eval_pred,
        "selection_metric_name": "AIC",
        "selection_metric": float(fitted.aic),
        "fit_seconds": float(fit_seconds),
        "aic": float(fitted.aic),
    }


def fit_arch_model(train_df: pd.DataFrame, eval_df: pd.DataFrame, model_type: str, p: int, q: int, o: int = 0) -> dict:
    if not ARCH_AVAILABLE:
        raise ImportError("The arch package is required for volatility benchmarks.")
    series = pd.Series(train_df[TARGET_COL].to_numpy(dtype=float) * 100.0)
    if len(series) < max(int(p), int(q), int(o)) + 5:
        raise ValueError("Not enough observations for the requested volatility specification.")
    if model_type == "ARCH":
        vol_mode = "ARCH"
        q_value = 0
        o_value = 0
    elif model_type == "GARCH":
        vol_mode = "GARCH"
        q_value = int(q)
        o_value = 0
    elif model_type == "GJR-GARCH":
        vol_mode = "GARCH"
        q_value = int(q)
        o_value = int(o)
    elif model_type == "EGARCH":
        vol_mode = "EGARCH"
        q_value = int(q)
        o_value = int(o)
    else:
        raise ValueError(model_type)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fit_start = time.perf_counter()
        model = arch_model(
            series,
            mean="AR",
            lags=1,
            vol=vol_mode,
            p=int(p),
            o=int(o_value),
            q=int(q_value),
            rescale=False,
        )
        fitted = model.fit(disp="off")
        fit_seconds = time.perf_counter() - fit_start
        eval_forecast = fitted.forecast(horizon=len(eval_df), reindex=False)

    eval_pred = np.asarray(eval_forecast.mean.iloc[-1], dtype=float).reshape(-1) / 100.0
    train_pred = np.full(len(train_df), np.nan, dtype=float)
    return {
        "y_train_pred": train_pred,
        "y_eval_pred": eval_pred,
        "selection_metric_name": "AIC",
        "selection_metric": float(fitted.aic),
        "fit_seconds": float(fit_seconds),
        "aic": float(fitted.aic),
    }


def build_prediction_rows(
    *,
    index_key: str,
    model_name: str,
    config_index: int,
    config_key_value: str,
    config_summary: str,
    config_json: str,
    eval_df: pd.DataFrame,
    eval_pred: np.ndarray,
    fit_seconds: float,
) -> list[dict]:
    rows = []
    preds = np.asarray(eval_pred, dtype=float).reshape(-1)
    for (_, row), y_pred in zip(eval_df.iterrows(), preds):
        reference_close = float(row["reference_close"])
        target_close = float(row["target_close"])
        y_true = float(row[TARGET_COL])
        rows.append(
            {
                "index_key": index_key,
                "model_name": model_name,
                "config_index": int(config_index),
                "config_key": config_key_value,
                "config_summary": config_summary,
                "config_json": config_json,
                "test_reference_date": pd.Timestamp(row["Date"]),
                "test_target_date": pd.Timestamp(row["target_date"]),
                "reference_close": reference_close,
                "target_close": target_close,
                "y_true": y_true,
                "y_pred": float(y_pred) if np.isfinite(y_pred) else np.nan,
                "actual_return": y_true if np.isfinite(y_true) else np.nan,
                "predicted_return": float(y_pred) if np.isfinite(y_pred) else np.nan,
                "actual_close": target_close if np.isfinite(target_close) else np.nan,
                "predicted_close": float(reference_close * (1.0 + y_pred)) if np.isfinite(y_pred) and np.isfinite(reference_close) else np.nan,
                "fit_seconds": float(fit_seconds),
            }
        )
    return rows


def select_best_configs(scores: pd.DataFrame) -> pd.DataFrame:
    picks = []
    for (index_key, model_name), group in scores.groupby(["index_key", "model"], as_index=False):
        valid = group.dropna(subset=["selection_metric"]).copy()
        if valid.empty:
            continue
        pick = valid.sort_values(["selection_metric", "eval_RMSE", "eval_MAE", "config_index"], ascending=[True, True, True, True], kind="stable").iloc[0]
        picks.append(pick.to_dict())
    picked = pd.DataFrame(picks)
    if picked.empty:
        return picked
    return picked.sort_values(["index_key", "model"]).reset_index(drop=True)


In [ ]:
BENCHMARK_CONFIGS: dict[str, list[dict]] = {
    "LinearRegression": [{}],
    "ARMA": [
        {"order": (0, 0, 1)},
        {"order": (1, 0, 0)},
        {"order": (1, 0, 1)},
        {"order": (1, 0, 2)},
        {"order": (2, 0, 1)},
        {"order": (2, 0, 2)},
    ],
    "ARIMA": [
        {"order": (0, 1, 0)},
        {"order": (0, 1, 1)},
        {"order": (1, 1, 0)},
        {"order": (1, 1, 1)},
        {"order": (1, 1, 2)},
        {"order": (2, 1, 1)},
        {"order": (2, 1, 2)},
    ],
    "ARCH": [{"p": 1, "q": 0}, {"p": 2, "q": 0}, {"p": 5, "q": 0}, {"p": 10, "q": 0}] if ARCH_AVAILABLE else [],
    "GARCH": [{"p": 1, "q": 1}, {"p": 1, "q": 2}, {"p": 2, "q": 1}, {"p": 2, "q": 2}] if ARCH_AVAILABLE else [],
    "GJR-GARCH": [{"p": 1, "o": 1, "q": 1}] if ARCH_AVAILABLE else [],
    "EGARCH": [{"p": 1, "o": 1, "q": 1}] if ARCH_AVAILABLE else [],
}


with timed_step("fit_chronological_benchmarks"):
    index_frames = load_index_frames()
    if FORCE_FRESH_RUN:
        reset_output_targets()

    existing_scores = load_progress_frame(SCORES_SUMMARY_PATH, key_cols=SCORE_KEY_COLS)
    completed_configs = completed_config_keys(existing_scores)
    total_configs_full = sum(len(configs) for configs in BENCHMARK_CONFIGS.values()) * len(index_frames)
    total_configs_pending = max(0, total_configs_full - len(completed_configs))

    with tqdm(total=total_configs_pending, desc="Benchmark fits") as progress_bar:
        for index_key, frame in index_frames.items():
            train_df, eval_df = split_train_eval(frame)

            for model_name, configs in BENCHMARK_CONFIGS.items():
                if not configs:
                    continue
                for config_index, config in enumerate(configs, start=1):
                    config_tuple = (index_key, model_name, int(config_index))
                    if config_tuple in completed_configs:
                        continue

                    progress_bar.set_postfix_str(f"{index_key} | {model_name}", refresh=False)
                    cfg_key = config_key(model_name, config)
                    cfg_summary = summarize_config(config)
                    cfg_json = json.dumps(config, sort_keys=True)
                    status = "completed"
                    error_message = ""
                    fit_seconds = np.nan
                    selection_metric_name = ""
                    selection_metric = np.nan
                    aic = np.nan
                    y_train_pred = np.full(len(train_df), np.nan, dtype=float)
                    y_eval_pred = np.full(len(eval_df), np.nan, dtype=float)

                    try:
                        if model_name == "LinearRegression":
                            result = fit_linear_model(train_df, eval_df)
                        elif model_name in {"ARMA", "ARIMA"}:
                            result = fit_arima_model(train_df, eval_df, order=tuple(config["order"]))
                        elif model_name in {"ARCH", "GARCH", "GJR-GARCH", "EGARCH"}:
                            result = fit_arch_model(
                                train_df,
                                eval_df,
                                model_type=model_name,
                                p=int(config["p"]),
                                q=int(config["q"]),
                                o=int(config.get("o", 0)),
                            )
                        else:
                            raise ValueError(model_name)

                        y_train_pred = np.asarray(result["y_train_pred"], dtype=float).reshape(-1)
                        y_eval_pred = np.asarray(result["y_eval_pred"], dtype=float).reshape(-1)
                        fit_seconds = float(result["fit_seconds"])
                        selection_metric_name = str(result["selection_metric_name"])
                        selection_metric = float(result["selection_metric"])
                        aic = float(result.get("aic", np.nan))
                    except Exception as exc:
                        status = "failed"
                        error_message = f"{type(exc).__name__}: {exc}"

                    train_metrics = regression_metrics(train_df[TARGET_COL].to_numpy(dtype=float), y_train_pred)
                    eval_metrics = regression_metrics(eval_df[TARGET_COL].to_numpy(dtype=float), y_eval_pred)

                    score_frame = pd.DataFrame(
                        [
                            {
                                "index_key": index_key,
                                "model": model_name,
                                "config_index": int(config_index),
                                "config_key": cfg_key,
                                "config_summary": cfg_summary,
                                "config_json": cfg_json,
                                "status": status,
                                "error_message": error_message,
                                "selection_metric_name": selection_metric_name,
                                "selection_metric": selection_metric,
                                "AIC": aic,
                                "train_MAE": train_metrics["MAE"],
                                "train_RMSE": train_metrics["RMSE"],
                                "train_R2": train_metrics["R2"],
                                "train_n": train_metrics["n_points"],
                                "train_directional_accuracy": directional_accuracy(train_df[TARGET_COL].to_numpy(dtype=float), y_train_pred),
                                "eval_MAE": eval_metrics["MAE"],
                                "eval_RMSE": eval_metrics["RMSE"],
                                "eval_R2": eval_metrics["R2"],
                                "eval_n": eval_metrics["n_points"],
                                "eval_directional_accuracy": directional_accuracy(eval_df[TARGET_COL].to_numpy(dtype=float), y_eval_pred),
                                "mean_fit_seconds": fit_seconds,
                                "fit_mode": "single_fit_chronological",
                                "target_definition": "one_step_return",
                            }
                        ]
                    )
                    if status == "completed":
                        prediction_frame = pd.DataFrame(
                            build_prediction_rows(
                                index_key=index_key,
                                model_name=model_name,
                                config_index=int(config_index),
                                config_key_value=cfg_key,
                                config_summary=cfg_summary,
                                config_json=cfg_json,
                                eval_df=eval_df,
                                eval_pred=y_eval_pred,
                                fit_seconds=fit_seconds,
                            )
                        )
                        append_frame(ALL_LOGS_PATH, prediction_frame)
                    append_frame(SCORES_SUMMARY_PATH, score_frame)
                    if status == "completed":
                        completed_configs.add(config_tuple)
                    progress_bar.update(1)

    scores_summary = load_progress_frame(SCORES_SUMMARY_PATH, key_cols=SCORE_KEY_COLS)
    prediction_log = load_progress_frame(ALL_LOGS_PATH, key_cols=PREDICTION_KEY_COLS)
    if not scores_summary.empty:
        scores_summary = scores_summary.sort_values(["index_key", "model", "config_index"]).reset_index(drop=True)
        scores_summary.to_csv(SCORES_SUMMARY_PATH, index=False)
    if not prediction_log.empty:
        prediction_log = prediction_log.sort_values(["index_key", "model_name", "config_index", "test_target_date"]).reset_index(drop=True)
        prediction_log.to_csv(ALL_LOGS_PATH, index=False)
        for (index_key, model_name), group in prediction_log.groupby(["index_key", "model_name"], as_index=False):
            group.to_csv(LOG_DIR / f"{index_key}_{model_name}.csv", index=False)

display(scores_summary.head(20))


In [ ]:
with timed_step("select_and_export_benchmark_summaries"):
    best_configs = select_best_configs(scores_summary)

    prediction_rows = []
    for _, best_row in best_configs.iterrows():
        subset = prediction_log.loc[
            prediction_log["index_key"].eq(best_row["index_key"])
            & prediction_log["model_name"].eq(best_row["model"])
            & prediction_log["config_key"].eq(best_row["config_key"])
        ].copy()
        prediction_path = PREDICTIONS_DIR / f"{best_row['index_key']}_{best_row['model']}_{best_row['config_key']}.csv"
        subset.to_csv(prediction_path, index=False)
        row = dict(best_row)
        row["prediction_path"] = str(prediction_path)
        prediction_rows.append(row)

    model_summary = pd.DataFrame(prediction_rows)
    if not model_summary.empty:
        model_summary = model_summary.sort_values(["index_key", "eval_RMSE", "eval_MAE"], ascending=[True, True, True]).reset_index(drop=True)
    best_configs.to_csv(BEST_CONFIGS_PATH, index=False)
    model_summary.to_csv(MODEL_SUMMARY_PATH, index=False)

timing_summary = pd.DataFrame(TIMING_ROWS).sort_values("elapsed_seconds", ascending=False).reset_index(drop=True)
timing_summary.to_csv(TIMING_SUMMARY_PATH, index=False)

print("Best configs")
display(best_configs)
print("Model summary")
display(model_summary)
print("Timing summary")
display(timing_summary)


In [ ]:
with timed_step("build_summary_figures"):
    summary_key = "index_key"
    summary_figure_path = FIG_DIR / "01_model_overview.png"
    if model_summary.empty:
        print("No completed runs available for summary figures.")
    else:
        avg_metrics = (
            model_summary.groupby("model", as_index=False)[["eval_MAE", "eval_RMSE", "eval_directional_accuracy", "mean_fit_seconds"]]
            .mean(numeric_only=True)
            .sort_values("eval_RMSE")
            .reset_index(drop=True)
        )
        heatmap_frame = model_summary.pivot(index=summary_key, columns="model", values="eval_RMSE").sort_index()

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].bar(avg_metrics["model"], avg_metrics["eval_MAE"], color="#2f6b82")
        axes[0].set_title("Average Evaluation MAE")
        axes[0].set_ylabel("Return MAE")
        axes[0].tick_params(axis="x", rotation=30)

        axes[1].bar(avg_metrics["model"], avg_metrics["eval_directional_accuracy"], color="#4f5866")
        axes[1].set_title("Average Directional Accuracy")
        axes[1].set_ylabel("Share Correct")
        axes[1].set_ylim(0.0, 1.0)
        axes[1].tick_params(axis="x", rotation=30)

        im = axes[2].imshow(heatmap_frame.to_numpy(dtype=float), aspect="auto", cmap="Blues")
        axes[2].set_title("Evaluation RMSE by Case")
        axes[2].set_xticks(range(len(heatmap_frame.columns)))
        axes[2].set_xticklabels(list(heatmap_frame.columns), rotation=30, ha="right")
        axes[2].set_yticks(range(len(heatmap_frame.index)))
        axes[2].set_yticklabels(list(heatmap_frame.index))
        fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

        fig.tight_layout()
        fig.savefig(summary_figure_path, dpi=200, bbox_inches="tight")
        plt.show()
        display(avg_metrics.round(6))
        display(heatmap_frame.round(6))
